In [79]:
using LowLevelFEM, LinearAlgebra, SparseArrays

In [80]:
structured_box_mesh(n=50)

mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=3, field=:u)

μ = mat.μ
λ = mat.λ
D = [λ+2μ λ λ 0 0 0; λ λ+2μ λ 0 0 0; λ λ λ+2μ 0 0 0; 0 0 0 μ 0 0; 0 0 0 0 μ 0; 0 0 0 0 0 μ]

6×6 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0      0.0      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0      0.0      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0      0.0      0.0
 0.0        0.0        0.0        76923.1      0.0      0.0
 0.0        0.0        0.0            0.0  76923.1      0.0
 0.0        0.0        0.0            0.0      0.0  76923.1

In [81]:
K = nothing
GC.gc(true)

@time K1 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), threads=2)

  3.384466 seconds (1.25 M allocations: 1.034 GiB, 2.20% gc time)


sparse([1, 2, 3, 169, 170, 171, 172, 173, 174, 1348  …  397803, 397804, 397805, 397806, 397948, 397949, 397950, 397951, 397952, 397953], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  397953, 397953, 397953, 397953, 397953, 397953, 397953, 397953, 397953, 397953], [940.1709401709376, 320.5128205128206, -320.51282051282095, 213.67521367521695, 160.2564102564115, 64.10256410256449, 213.6752136752119, -64.10256410256422, -160.25641025641008, -427.35042735042634  …  -42.735042735041986, 4.831690603168681e-13, -1.7621459846850485e-12, 854.70085470086, 8.810729923425242e-13, 1.7053025658242404e-13, 854.700854700857, 1.6484591469634324e-12, 5.684341886080802e-14, 7521.367521367541], 397953, 397953)

In [82]:
@time Kpattern = build_csc_pattern(Pu, Pu);

  1.281156 seconds (1.25 M allocations: 809.612 MiB, 6.57% gc time)


In [83]:
K = nothing
GC.gc(true)

@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), threads=2, csc_matrix=Kpattern)

  3.055853 seconds (380 allocations: 249.095 MiB)


sparse([1, 2, 3, 169, 170, 171, 172, 173, 174, 1348  …  397803, 397804, 397805, 397806, 397948, 397949, 397950, 397951, 397952, 397953], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  397953, 397953, 397953, 397953, 397953, 397953, 397953, 397953, 397953, 397953], [940.1709401709376, 320.5128205128206, -320.51282051282095, 213.67521367521695, 160.2564102564115, 64.10256410256449, 213.6752136752119, -64.10256410256422, -160.25641025641008, -427.35042735042634  …  -42.735042735041986, 4.831690603168681e-13, -1.7621459846850485e-12, 854.70085470086, 8.810729923425242e-13, 1.7053025658242404e-13, 854.700854700857, 1.6484591469634324e-12, 5.684341886080802e-14, 7521.367521367541], 397953, 397953)

In [84]:
norm(K1.A - K2.A) / norm(K1.A)

0.0

In [85]:
println("size: ", size(K1.A))
println("nnz: ", nnz(K1.A))
#norm(K1.A)

size: (397953, 397953)
nnz: 30959679


In [86]:
K = nothing
GC.gc(true)

@time K1 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), Γ="left", threads=2)

  0.043608 seconds (449.98 k allocations: 38.798 MiB)


sparse([1, 169, 172, 8845, 2, 3, 170, 171, 173, 174  …  464, 465, 8840, 8841, 8843, 8844, 8987, 8988, 8990, 8991], [1, 1, 1, 1, 2, 2, 2, 2, 2, 2  …  8991, 8991, 8991, 8991, 8991, 8991, 8991, 8991, 8991, 8991], [51282.051282051325, -12820.512820512826, -12820.512820512722, -25641.02564102577, 115384.61538461545, -48076.923076923296, 19230.769230769256, 9615.384615384834, -76923.07692307665, -9615.384615384723  …  -6.184563972055912e-11, 38461.5384615385, -48076.923076923034, -57692.30769230766, 9.276845958083868e-11, -153846.15384615381, -5.093170329928398e-11, 38461.5384615386, -1.5279510989785194e-10, 461538.46153846156], 397953, 397953)

In [87]:
@time Kpattern = build_csc_pattern(Pu, Pu, Γ="left");

  0.004740 seconds (142.90 k allocations: 12.613 MiB)


In [88]:
K = nothing
GC.gc(true)

@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), Γ="left", threads=2, csc_matrix=Kpattern)

  0.031916 seconds (307.10 k allocations: 26.186 MiB)


sparse([1, 2, 3, 169, 170, 171, 172, 173, 174, 8845  …  8841, 8842, 8843, 8844, 8986, 8987, 8988, 8989, 8990, 8991], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  8991, 8991, 8991, 8991, 8991, 8991, 8991, 8991, 8991, 8991], [51282.051282051325, 0.0, 0.0, -12820.512820512826, 0.0, 0.0, -12820.512820512722, 0.0, 0.0, -25641.02564102577  …  -57692.30769230766, 0.0, 9.276845958083868e-11, -153846.15384615381, 0.0, -5.093170329928398e-11, 38461.5384615386, 0.0, -1.5279510989785194e-10, 461538.46153846156], 397953, 397953)

In [89]:
norm(K1.A - K2.A) / norm(K1.A)

0.0

In [90]:
println("size: ", size(K1.A))
println("nnz: ", nnz(K1.A))
#norm(K1.A)

size: (397953, 397953)
nnz: 113559
